# v2: A — MoCo queue supervised contrastive learning

Reuse existing prepared_data and splits. Two stochastic dropout views, EMA key encoder, training-only labeled FIFO.


In [ ]:
RUN_TRAINING = True  # 已有检查点时可设为 False，仅运行后续评估。


In [ ]:
import math
import random
import subprocess
import sys
import json
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/prepared_data")
OUTPUT_DIR = Path("/content/drive/MyDrive/contrastive_outputs_v2")


## 1. 环境与依赖


In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers==4.52.4", "sentencepiece>=0.2", "safetensors>=0.4"])
# 保留 Colab 预装的 GPU 版 PyTorch，不替换为 CPU 版。


## 2. 数据路径


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3. 训练配置


In [ ]:
config = {
    "data_run": str(DATA_DIR),
    "split": "grouped_multilingual",
    "model_name": "FacebookAI/xlm-roberta-base",
    "task": "A",  # A、B、C；每次独立训练一个子任务
    "max_length": 512,
    "dropout": 0.1,
    "projection_dim": 128,
    "epochs": 3,
    "batch_size": 4,
    "eval_batch_size": 8,
    "gradient_accumulation": 4,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "seed": 42,
    "device": "cuda",
    "contrastive_weight": 0.1,
    "temperature": 0.1,
    "queue_size": 4096,
    "momentum": 0.999,
    "identity_threshold": 0.5,  # 仅用于 C 的身份预测
    "mixed_precision": True,
}
RUN_NAME = f"{config['task']}_moco_scl_seed{config['seed']}"


## 4. 模型与训练函数


### 4.1 导入与标签映射


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

IDENTITIES = ['l', 'g', 'b', 't', 'q', 'i', 'a', 'nb', 'lgbtqia+']



### 4.2 数据读取与输入编码


In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

In [ ]:
def load_split(data_run, split_name):
    root = Path(data_run)
    records = {}
    for path in sorted((root / 'processed').glob('*.jsonl')):
        for line in path.read_text(encoding='utf-8').splitlines():
            row = json.loads(line)
            records[row['id']] = row
    split = read_json(root / 'splits' / f'{split_name}.json')
    train = [records[i] for i in split['train_ids']]
    val = [records[i] for i in split['validation_ids']]
    return train, val


In [ ]:
def encode_record(row, tokenizer, config):
    text = f"[COMMENT] {row['comment']} [TITLE] {row['title']} [DESCRIPTION] {row['description']}"
    return tokenizer(text, truncation=True, max_length=config['max_length'])


In [ ]:
class TextDataset(Dataset):
    def __init__(self, rows, tokenizer, config):
        self.rows = rows
        self.inputs = [encode_record(row, tokenizer, config) for row in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return {'inputs': self.inputs[index], 'row': self.rows[index]}


In [ ]:
class Collator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, items):
        rows = [item['row'] for item in items]
        batch = {
            'inputs': self.tokenizer.pad([item['inputs'] for item in items], padding=True, return_tensors='pt'),
            'rows': rows,
        }
        for field, dtype in [('stereotype_label', torch.long), ('hate_label', torch.long),
                             ('scope_label', torch.long), ('identity_labels', torch.float32),
                             ('target_loss_mask', torch.bool)]:
            batch[field] = torch.tensor([row[field] for row in rows], dtype=dtype)
        return batch


### 4.3 预测解码与评估指标


In [ ]:
def classification_metrics(gold, pred, labels):
    scores = []
    for label in labels:
        tp = sum(g == label and p == label for g, p in zip(gold, pred))
        fp = sum(g != label and p == label for g, p in zip(gold, pred))
        fn = sum(g == label and p != label for g, p in zip(gold, pred))
        denominator = 2*tp+fp+fn
        scores.append({'label': label, 'support': sum(g == label for g in gold),
                       'precision': tp/(tp+fp) if tp+fp else 0.,
                       'recall': tp/(tp+fn) if tp+fn else 0.,
                       'f1': 2*tp/denominator if denominator else 0.})
    return {'n': len(gold), 'accuracy': sum(g == p for g, p in zip(gold, pred))/len(gold) if gold else None,
            'macro_f1': sum(s['f1'] for s in scores)/len(labels), 'per_class': scores}


In [ ]:
def decode(outputs, task, threshold=.5):
    if task == 'A':
        return [{'stereotype': 'yes' if label else 'no'} for label in outputs['a'].argmax(-1).tolist()]
    if task == 'B':
        return [{'hate_pred': label} for label in outputs['b'].argmax(-1).tolist()]
    scopes = outputs['scope'].argmax(-1).tolist()
    probabilities = outputs['identity'].sigmoid().tolist()
    predictions = []
    for scope, values in zip(scopes, probabilities):
        identities = [int(value >= threshold) for value in values]
        if not any(identities):
            identities[max(range(len(values)), key=values.__getitem__)] = 1
        target = ('group' if scope == 0 else 'individual') + '_' + ','.join(
            name for name, active in zip(IDENTITIES, identities) if active)
        predictions.append({'scope_pred': scope, 'identity_pred': identities, 'target': target})
    return predictions


In [ ]:
def evaluate_metrics(rows, predictions, task):
    if task == 'A':
        return classification_metrics([r['stereotype_label'] for r in rows],
                                      [int(p['stereotype'] == 'yes') for p in predictions], [0, 1])
    if task == 'B':
        return classification_metrics([r['hate_label'] for r in rows],
                                      [p['hate_pred'] for p in predictions], [0, 1, 2])
    scope = classification_metrics([r['scope_label'] for r in rows],
                                   [p['scope_pred'] for p in predictions], [0, 1])
    identities = []
    for i, name in enumerate(IDENTITIES):
        score = classification_metrics([r['identity_labels'][i] for r in rows],
                                       [p['identity_pred'][i] for p in predictions], [1])['per_class'][0]
        identities.append(dict(score, label=name))
    # Macro over the nine positive-label F1 scores (multilabel binary vectors).
    identity_macro_f1 = sum(score['f1'] for score in identities) / len(IDENTITIES)
    official_score = 0.5 * scope['macro_f1'] + 0.5 * identity_macro_f1
    return {'n': len(rows), 'scope': scope, 'identity_per_label': identities,
            'official_score': official_score,
            'identity_macro_f1': identity_macro_f1,
            'target_exact_match': sum(r['target'] == p['target'] for r, p in zip(rows, predictions)) / len(rows)}


### 4.4 子任务模型与 SCL 损失


In [ ]:
class TaskModel(nn.Module):
    def __init__(self, encoder, task, dropout=.1, projection_dim=128):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        sizes = {'A': {'a': 2}, 'B': {'b': 3}, 'C': {'scope': 2, 'identity': len(IDENTITIES)}}
        self.heads = nn.ModuleDict({name: nn.Linear(hidden, size) for name, size in sizes[task].items()})
        self.projection = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, projection_dim))

    def forward(self, **inputs):
        sequence = self.encoder(**inputs).last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).to(sequence.dtype)
        pooled = (sequence * mask).sum(1) / mask.sum(1).clamp_min(1)
        hidden = self.dropout(pooled)
        outputs = {name: head(hidden) for name, head in self.heads.items()}
        outputs['projection'] = F.normalize(self.projection(hidden), dim=-1)
        return outputs


In [ ]:
def classification_loss(outputs, batch, task):
    if task == 'A':
        return F.cross_entropy(outputs['a'], batch['stereotype_label'])
    if task == 'B':
        return F.cross_entropy(outputs['b'], batch['hate_label'])
    scope = F.cross_entropy(outputs['scope'], batch['scope_label'])
    identity = F.binary_cross_entropy_with_logits(outputs['identity'], batch['identity_labels'])
    return scope + identity


In [ ]:
def contrastive_labels(batch, task):
    if task == 'A':
        return batch['stereotype_label'].unsqueeze(1)
    if task == 'B':
        return batch['hate_label'].unsqueeze(1)
    # C：对象类型和完整身份标签集合相同，才视为正例。
    return torch.cat([batch['scope_label'].unsqueeze(1), batch['identity_labels'].long()], dim=1)



In [ ]:
import copy

class MomentumQueue(nn.Module):
    """Training-only momentum encoder and labeled FIFO; checkpoints retain query model only."""
    def __init__(self, model, size, label_dim, momentum):
        super().__init__()
        if size <= 0 or not 0 <= momentum < 1:
            raise ValueError('queue_size must be positive and momentum must be in [0, 1)')
        self.key = copy.deepcopy(model).requires_grad_(False)
        self.momentum = momentum
        self.register_buffer('features', torch.zeros(size, model.projection[-1].out_features))
        self.register_buffer('labels', torch.zeros(size, label_dim, dtype=torch.long))
        self.register_buffer('ids', torch.full((size,), -1, dtype=torch.long))
        self.ptr = 0
        self.count = 0

    @torch.no_grad()
    def update(self, model):
        for key, query in zip(self.key.parameters(), model.parameters()):
            key.mul_(self.momentum).add_(query, alpha=1-self.momentum)

    @torch.no_grad()
    def enqueue(self, features, labels, ids):
        # Sequential ring writes also support a batch larger than the queue.
        for feature, label, sample_id in zip(features.detach(), labels, ids):
            self.features[self.ptr].copy_(feature)
            self.labels[self.ptr].copy_(label)
            self.ids[self.ptr].copy_(sample_id)
            self.ptr = (self.ptr + 1) % len(self.features)
            self.count = min(self.count + 1, len(self.features))


def queue_scl_loss(queries, keys, labels, ids, memory, temperature):
    if temperature <= 0:
        raise ValueError('temperature must be positive')
    # The paired dropout view is always a positive, even for rare C labels.
    candidates = torch.cat([keys.detach().float(), memory.features[:memory.count].detach().clone()], 0)
    candidate_labels = torch.cat([labels, memory.labels[:memory.count]], 0)
    candidate_ids = torch.cat([ids, memory.ids[:memory.count]], 0)
    paired = torch.zeros(len(queries), len(candidates), dtype=torch.bool, device=queries.device)
    paired[:, :len(keys)] = torch.eye(len(keys), dtype=torch.bool, device=queries.device)
    # Exclude stale views of the same sample; retain the current paired key.
    allowed = (ids[:, None] != candidate_ids[None, :]) | paired
    positive = (labels[:, None, :] == candidate_labels[None, :, :]).all(-1) & allowed
    logits = F.normalize(queries.float(), dim=-1) @ F.normalize(candidates, dim=-1).T / temperature
    log_probs = logits - torch.logsumexp(logits.masked_fill(~allowed, float('-inf')), dim=1, keepdim=True)
    return -(log_probs.masked_fill(~positive, 0).sum(1) / positive.sum(1).clamp_min(1)).mean()


### 4.5 训练工具与评估


In [ ]:
def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


In [ ]:
def move(batch, device):
    return {k: ({ik: iv.to(device) for ik, iv in v.items()} if k == 'inputs'
                else v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


In [ ]:
@torch.no_grad()
def evaluate(model, loader, config, device):
    model.eval()
    rows, predictions = [], []
    for raw in loader:
        batch = move(raw, device)
        outputs = model(**batch['inputs'])
        predictions.extend(decode(outputs, config['task'], config['identity_threshold']))
        rows.extend(raw['rows'])
    return evaluate_metrics(rows, predictions, config['task'])


In [ ]:
def selection_score(metrics, task):
    return metrics['official_score'] if task == 'C' else metrics['macro_f1']


### 4.6 最佳模型保存


In [ ]:
def save_checkpoint(model, tokenizer, config, root):
    root.mkdir(exist_ok=True)
    model.encoder.config.save_pretrained(root/'encoder_config')
    tokenizer.save_pretrained(root/'tokenizer')
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, root/'model.pt')
    write_json(root/'config.json', config)



### 4.7 SCL 训练循环


In [ ]:
def train(config, train_rows, val_rows, run_name):
    config = dict(config)
    root = OUTPUT_DIR / 'model_runs' / run_name
    seed_everything(config['seed'])
    if config['task'] == 'C':
        train_rows = [r for r in train_rows if r['target_loss_mask']]
        val_rows = [r for r in val_rows if r['target_loss_mask']]
    device = torch.device(config["device"])
    root.mkdir(parents=True, exist_ok=True)
    write_json(root/'config.json', config)
    cache = OUTPUT_DIR/'cache'/'huggingface'
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'], cache_dir=cache)
    encoder = AutoModel.from_pretrained(config['model_name'], cache_dir=cache)
    encoder.to(device)
    train_data = TextDataset(train_rows, tokenizer, config)
    val_data = TextDataset(val_rows, tokenizer, config)
    model = TaskModel(encoder, config['task'], config['dropout'], config['projection_dim']).to(device)
    if not train_rows or not val_rows:
        raise ValueError('Training and validation splits must be nonempty')
    sample_ids = {row['id']: i for i, row in enumerate(train_rows)}
    memory = MomentumQueue(model, config['queue_size'], 10 if config['task'] == 'C' else 1, config['momentum']).to(device)
    generator = torch.Generator().manual_seed(config['seed'])
    loader = DataLoader(train_data, batch_size=config['batch_size'], shuffle=True, generator=generator,
                        collate_fn=Collator(tokenizer), num_workers=0)
    validation = DataLoader(val_data, batch_size=config['eval_batch_size'], collate_fn=Collator(tokenizer), num_workers=0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    steps = math.ceil(len(loader)/config['gradient_accumulation'])*config['epochs']
    scheduler = get_linear_schedule_with_warmup(optimizer, int(steps*config['warmup_ratio']), steps)
    amp_enabled = config.get('mixed_precision', False) and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
    best, history = -1., []
    for epoch in range(config['epochs']):
        model.train()
        memory.key.train()  # Independent dropout gives the second text view.
        optimizer.zero_grad()
        total, supervised_total, contrast_total = 0., 0., 0.
        for index, raw in enumerate(loader):
            batch = move(raw, device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
                outputs = model(**batch['inputs'])
                supervised = classification_loss(outputs, batch, config['task'])
                labels = contrastive_labels(batch, config['task'])
                ids = torch.tensor([sample_ids[r['id']] for r in raw['rows']], device=device)
                with torch.no_grad():
                    keys = memory.key(**batch['inputs'])['projection']
                contrast = queue_scl_loss(outputs['projection'], keys, labels, ids, memory, config['temperature'])
                loss = supervised + config['contrastive_weight']*contrast
            if not torch.isfinite(loss):
                raise ValueError('训练损失出现非有限数值')
            # 正确缩放最后一个不足完整大小的梯度累积窗口。
            window_start = (index//config['gradient_accumulation'])*config['gradient_accumulation']
            window_size = min(config['gradient_accumulation'], len(loader)-window_start)
            scaler.scale(loss/window_size).backward()
            memory.enqueue(keys, labels, ids)
            total += loss.item()
            supervised_total += supervised.item()
            contrast_total += contrast.item()
            if (index+1) % config['gradient_accumulation'] == 0 or index+1 == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                    memory.update(model)
                optimizer.zero_grad()
        metrics = evaluate(model, validation, config, device)
        score = selection_score(metrics, config['task'])
        record = {'epoch': epoch+1, 'loss': total/len(loader), 'classification_loss': supervised_total/len(loader),
                  'contrastive_loss': contrast_total/len(loader), 'queue_count': memory.count,
                  'selection_score': score, 'validation': metrics}
        history.append(record)
        write_json(root/'history.json', history)
        metric_name = "C 官方综合 F1" if config["task"] == "C" else "Macro-F1"
        print(f"训练轮次：{epoch+1}，总损失：{record['loss']:.6f}，{metric_name}：{score:.6f}", flush=True)
        if score > best:
            best = score
            save_checkpoint(model, tokenizer, config, root/'best')
            write_json(root/'best_metrics.json', metrics)
    return root


## 5. 读取数据划分


In [ ]:
train_rows, validation_rows = load_split(config['data_run'], config['split'])
print("训练样本：", len(train_rows), "验证样本：", len(validation_rows))


## 6. 执行训练


In [ ]:
if RUN_TRAINING:
    model_run = train(config, train_rows, validation_rows, RUN_NAME)
    print("模型保存目录：", model_run)
else:
    model_run = OUTPUT_DIR / "model_runs" / RUN_NAME


## 7. 最佳模型评估与无标签推理

默认评估本次训练的最佳检查点。也可跳过训练，修改 CHECKPOINT 后运行。C 的无标签推理需要同种子的 B 检查点。


In [ ]:
import csv
from transformers import AutoConfig
CHECKPOINT = model_run / 'best'
EVALUATION_SPLIT = config['split']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = config.get('eval_batch_size', 8)
RUN_EVALUATION = True


In [ ]:
def evaluation_read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


In [ ]:
def evaluation_encode_record(row, tokenizer, evaluation_config):
    text = f'[COMMENT] {row['comment']} [TITLE] {row['title']} [DESCRIPTION] {row['description']}'
    return tokenizer(text, truncation=True, max_length=evaluation_config['max_length'])


In [ ]:
class evaluation_TextDataset(Dataset):

    def __init__(self, rows, tokenizer, evaluation_config):
        self.rows = rows
        self.inputs = [evaluation_encode_record(row, tokenizer, evaluation_config) for row in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return {'inputs': self.inputs[index], 'row': self.rows[index]}


In [ ]:
class evaluation_PredictionCollator:

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, items):
        return {'inputs': self.tokenizer.pad([item['inputs'] for item in items], padding=True, return_tensors='pt'), 'rows': [item['row'] for item in items]}


In [ ]:
class evaluation_TaskModel(nn.Module):

    def __init__(self, encoder, task, dropout=0.1, projection_dim=128):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        sizes = {'A': {'a': 2}, 'B': {'b': 3}, 'C': {'scope': 2, 'identity': len(IDENTITIES)}}
        self.heads = nn.ModuleDict({name: nn.Linear(hidden, size) for name, size in sizes[task].items()})
        self.projection = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, projection_dim))

    def forward(self, **inputs):
        sequence = self.encoder(**inputs).last_hidden_state
        mask = inputs['attention_mask'].unsqueeze(-1).to(sequence.dtype)
        pooled = (sequence * mask).sum(1) / mask.sum(1).clamp_min(1)
        hidden = self.dropout(pooled)
        outputs = {name: head(hidden) for name, head in self.heads.items()}
        outputs['projection'] = F.normalize(self.projection(hidden), dim=-1)
        return outputs


In [ ]:
def evaluation_load_checkpoint(root, device):
    evaluation_config = evaluation_read_json(root / 'config.json')
    encoder_config = AutoConfig.from_pretrained(root / 'encoder_config', local_files_only=True)
    encoder = AutoModel.from_config(encoder_config)
    model = evaluation_TaskModel(encoder, evaluation_config['task'], evaluation_config['dropout'], evaluation_config['projection_dim'])
    model.load_state_dict(torch.load(root / 'model.pt', map_location='cpu', weights_only=True))
    tokenizer = AutoTokenizer.from_pretrained(root / 'tokenizer', local_files_only=True)
    return (model.to(device).eval(), tokenizer, evaluation_config)


In [ ]:
def evaluation_load_evaluation_rows(root, split_name, task):
    records = {}
    for path in sorted((root / 'processed').glob('*.jsonl')):
        for line in path.read_text(encoding='utf-8').splitlines():
            row = json.loads(line)
            records[row['id']] = row
    split = evaluation_read_json(root / 'splits' / f'{split_name}.json')
    ids = split['evaluation_ids'] if split_name == 'persian_external' else split['validation_ids']
    rows = [records[sample_id] for sample_id in ids]
    if task == 'C':
        rows = [row for row in rows if row['target_loss_mask']]
    return rows


In [ ]:
def evaluation_classification_metrics(gold, pred, labels):
    scores = []
    for label in labels:
        tp = sum((g == label and p == label for g, p in zip(gold, pred)))
        fp = sum((g != label and p == label for g, p in zip(gold, pred)))
        fn = sum((g == label and p != label for g, p in zip(gold, pred)))
        denominator = 2 * tp + fp + fn
        scores.append({'label': label, 'support': sum((g == label for g in gold)), 'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0, 'f1': 2 * tp / denominator if denominator else 0.0})
    return {'n': len(gold), 'accuracy': sum((g == p for g, p in zip(gold, pred))) / len(gold) if gold else None, 'macro_f1': sum((s['f1'] for s in scores)) / len(labels), 'per_class': scores}


In [ ]:
def evaluation_decode(outputs, task, threshold=0.5):
    if task == 'A':
        return [{'stereotype': 'yes' if label else 'no'} for label in outputs['a'].argmax(-1).tolist()]
    if task == 'B':
        return [{'hate_pred': label} for label in outputs['b'].argmax(-1).tolist()]
    scopes = outputs['scope'].argmax(-1).tolist()
    probabilities = outputs['identity'].sigmoid().tolist()
    predictions = []
    for scope, values in zip(scopes, probabilities):
        identities = [int(value >= threshold) for value in values]
        if not any(identities):
            identities[max(range(len(values)), key=values.__getitem__)] = 1
        target = ('group' if scope == 0 else 'individual') + '_' + ','.join((name for name, active in zip(IDENTITIES, identities) if active))
        predictions.append({'scope_pred': scope, 'identity_pred': identities, 'target': target})
    return predictions


In [ ]:
def evaluation_evaluate_metrics(rows, predictions, task):
    if task == 'A':
        return evaluation_classification_metrics([r['stereotype_label'] for r in rows], [int(p['stereotype'] == 'yes') for p in predictions], [0, 1])
    if task == 'B':
        return evaluation_classification_metrics([r['hate_label'] for r in rows], [p['hate_pred'] for p in predictions], [0, 1, 2])
    scope = evaluation_classification_metrics([r['scope_label'] for r in rows], [p['scope_pred'] for p in predictions], [0, 1])
    identities = []
    for i, name in enumerate(IDENTITIES):
        score = evaluation_classification_metrics([r['identity_labels'][i] for r in rows], [p['identity_pred'][i] for p in predictions], [1])['per_class'][0]
        identities.append(dict(score, label=name))
    identity_macro_f1 = sum((score['f1'] for score in identities)) / len(IDENTITIES)
    official_score = 0.5 * scope['macro_f1'] + 0.5 * identity_macro_f1
    return {'n': len(rows), 'scope': scope, 'identity_per_label': identities, 'official_score': official_score, 'identity_macro_f1': identity_macro_f1, 'target_exact_match': sum((r['target'] == p['target'] for r, p in zip(rows, predictions))) / len(rows)}


In [ ]:
@torch.no_grad()
def evaluation_predict_rows(model, tokenizer, evaluation_config, rows, device):
    dataset = evaluation_TextDataset(rows, tokenizer, evaluation_config)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, collate_fn=evaluation_PredictionCollator(tokenizer))
    predictions = []
    for batch in loader:
        inputs = {name: value.to(device) for name, value in batch['inputs'].items()}
        outputs = model(**inputs)
        predictions.extend(evaluation_decode(outputs, evaluation_config['task'], evaluation_config['identity_threshold']))
    return predictions

def evaluation_predict_test_rows(model, tokenizer, evaluation_config, rows, device, checkpoint):
    if evaluation_config['task'] != 'C':
        return evaluation_predict_rows(model, tokenizer, evaluation_config, rows, device)
    b_checkpoint = checkpoint.parent.parent / f'B_moco_scl_seed{evaluation_config['seed']}' / 'best'
    b_model, b_tokenizer, b_config = evaluation_load_checkpoint(b_checkpoint, device)
    hate_predictions = evaluation_predict_rows(b_model, b_tokenizer, b_config, rows, device)
    del b_model
    hateful_indices = [i for i, prediction in enumerate(hate_predictions) if prediction['hate_pred'] != 0]
    hateful_rows = [rows[i] for i in hateful_indices]
    targets = evaluation_predict_rows(model, tokenizer, evaluation_config, hateful_rows, device)
    predictions = [{'target': 'none'} for _ in rows]
    for index, target in zip(hateful_indices, targets):
        predictions[index] = target
    return predictions

def evaluation_write_predictions(path, rows, predictions, task):
    column = {'A': 'stereotype', 'B': 'hate_speech', 'C': 'target'}[task]
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['StereoQueerEval_id', column], delimiter='\t')
        writer.writeheader()
        for row, prediction in zip(rows, predictions):
            value = HATE[prediction['hate_pred']] if task == 'B' else prediction[column]
            writer.writerow({'StereoQueerEval_id': row['id'], column: value})


In [ ]:
if RUN_EVALUATION:
    device = torch.device(DEVICE)
    model, tokenizer, evaluation_config = evaluation_load_checkpoint(CHECKPOINT, device)
    rows = evaluation_load_evaluation_rows(DATA_DIR, EVALUATION_SPLIT, evaluation_config['task'])
    predictions = evaluation_predict_rows(model, tokenizer, evaluation_config, rows, device)
    metrics = evaluation_evaluate_metrics(rows, predictions, evaluation_config['task'])
    result_dir = OUTPUT_DIR / 'evaluations' / CHECKPOINT.parent.name / EVALUATION_SPLIT
    result_dir.mkdir(parents=True, exist_ok=True)
    (result_dir / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    evaluation_write_predictions(result_dir / 'predictions.tsv', rows, predictions, evaluation_config['task'])
    if evaluation_config['task'] == 'C':
        print(f'样本：{len(rows)}，C 官方综合 F1：{metrics['official_score']:.6f}，对象完全匹配率：{metrics['target_exact_match']:.6f}，范围 Macro-F1：{metrics['scope']['macro_f1']:.6f}，身份 Macro-F1：{metrics['identity_macro_f1']:.6f}')
    else:
        print(f'样本：{len(rows)}，Macro-F1：{metrics['macro_f1']:.6f}，准确率：{metrics['accuracy']:.6f}')
    print('评估结果目录：', result_dir)


In [ ]:
if not RUN_EVALUATION:
    device = torch.device(DEVICE)
    model, tokenizer, evaluation_config = evaluation_load_checkpoint(CHECKPOINT, device)
RUN_PREDICTION = False
INPUT_PATH = Path('/content/drive/MyDrive/test.tsv')
PREDICTION_PATH = OUTPUT_DIR / 'predictions' / CHECKPOINT.parent.name / 'test.tsv'
if RUN_PREDICTION:
    with INPUT_PATH.open(encoding='utf-8-sig', newline='') as handle:
        test_rows = [{'id': row['StereoQueerEval_id'], 'comment': row['yt_comment'], 'title': row['yt_title'], 'description': row['yt_description']} for row in csv.DictReader(handle, delimiter='\t')]
    test_predictions = evaluation_predict_test_rows(model, tokenizer, evaluation_config, test_rows, device, CHECKPOINT)
    evaluation_write_predictions(PREDICTION_PATH, test_rows, test_predictions, evaluation_config['task'])
    print('预测文件：', PREDICTION_PATH)
